# Boolean circuits with an autoregressive Transformer
## Generate a symbolic reasoning trace, then the answer

This tutorial implements the Transformer design supplied with this notebook:
a **single causal self-attention layer, four heads, residual connections, one
hidden ReLU MLP, and no LayerNorm**. The learned parameters are token and position
embeddings, attention projections, MLP weights, and a vocabulary readout.

The model generates a visible **symbolic reasoning trace**: a gate and a four-bit
state at each step. This is task-specific chain-of-thought supervision, not a
natural-language explanation. At inference, each new token comes from the model's
logits and is appended to its input; gold intermediate states are never supplied.

We compare:

1. A **fixed hand-coded causal Transformer** that constructs the correct trace.
2. An **outcome-trained Transformer** that learns a short answer continuation.
3. A **process-trained Transformer** that learns a gate/state trace and answer.
4. A **fixed hand-coded outcome Transformer** that computes the answer internally
   across one block per gate and emits only the final-answer continuation.

The two learned models use identical architecture, initial parameters, circuit
examples, minibatch indices, and update count. Process examples have longer
continuations and more target tokens, so this is not a compute-matched experiment.
The hand-coded reference uses a larger, structured representation and MLP; it is
not a proof that the narrower learned model can express the same construction.

Run top to bottom in the project's Python environment. PyTorch, NumPy, pandas,
Matplotlib, and IPython are required. CPU works; the full training and animation
can take several minutes. Reduce `STEPS` for a smoke run, without expecting the
same accuracy. The functions are grouped into task, tokenization, model,
training, evaluation, and visualization cells.

In [69]:
import copy
import math
import random
from dataclasses import dataclass
from itertools import permutations
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
from IPython.display import HTML, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_BITS = 4
N_STATES = 2 ** N_BITS
DEPTH = 4
TRAIN_SIZE = 8_000
TEST_SIZE = 300
STEPS = 10_000
BATCH_SIZE = 128
LR = 2e-3
D_MODEL = 96
N_HEADS = 4
D_FF = 192
DATA_SEED, TEST_SEED, MODEL_SEED, BATCH_SEED = 123, 9000, 42, 2026
CHECKPOINTS = sorted({step for step in [0, 100, 250, 500, STEPS] if step <= STEPS})
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False,
                     "axes.spines.right": False})
print(f"Device: {DEVICE}; training updates per model: {STEPS}")

Device: cuda; training updates per model: 10000


## 1. Exact circuits provide the training labels

Index 0 is the **leftmost** bit. The generator chooses a gate family uniformly,
then distinct indices within that family. There are 52 gate strings; symmetric
spellings such as `s01` and `s10` are retained as separate tokens.

| Gate | Operation | Example |
|---|---|---|
| `x0` | Flip bit 0 | `0000 → 1000` |
| `c01` | Flip bit 1 when bit 0 is 1 | `1000 → 1100` |
| `s01` | Swap bits 0 and 1 | `1000 → 0100` |
| `t012` | Flip bit 2 when bits 0 and 1 are both 1 | `1100 → 1110` |

These Python functions create labels and initialize the fixed reference weights.
The learned Transformer's forward pass and generation loop never call them.

In [70]:
def bits(value):
    """Decode an integer; index 0 is the leftmost bit."""
    return [int(bit) for bit in f"{value:0{N_BITS}b}"]


def state_text(state):
    """Format a list of bits as a readable state such as '1000'."""
    return "".join(map(str, state))


def make_gate_names(n_bits):
    """List each legal gate string in a stable order."""
    gates = [f"x{i}" for i in range(n_bits)]
    for operation in ("c", "s"):
        gates.extend(f"{operation}{i}{j}" for i, j in permutations(range(n_bits), 2))
    gates.extend(f"t{i}{j}{k}" for i, j, k in permutations(range(n_bits), 3))
    return gates


def sample_gate(rng):
    """Choose a family uniformly, then choose its distinct bit indices."""
    operation = rng.choice("xcst")
    arity = {"x": 1, "c": 2, "s": 2, "t": 3}[operation]
    indices = rng.sample(range(N_BITS), arity)
    return operation + "".join(map(str, indices))


def apply_gate(state, gate):
    """Apply one reversible gate without modifying the input state."""
    next_state = state.copy()
    operation = gate[0]
    indices = list(map(int, gate[1:]))
    if operation == "x":
        target, = indices
        next_state[target] ^= 1
    elif operation == "c":
        control, target = indices
        next_state[target] ^= next_state[control]
    elif operation == "s":
        left, right = indices
        next_state[left], next_state[right] = next_state[right], next_state[left]
    elif operation == "t":
        control_a, control_b, target = indices
        next_state[target] ^= next_state[control_a] & next_state[control_b]
    else:
        raise ValueError(f"Unknown gate: {gate}")
    return next_state


def phi(state_int, gate):
    """Integer version of the exact Boolean transition rule."""
    return int(state_text(apply_gate(bits(state_int), gate)), 2)


def sample_example(rng, depth):
    """Return the start, gate names, and gold state after every gate."""
    start = rng.randrange(N_STATES)
    gates = [sample_gate(rng) for _ in range(depth)]
    states = []
    current = start
    for gate in gates:
        current = phi(current, gate)
        states.append(current)
    return start, gates, states


GATES = make_gate_names(N_BITS)
gate_to_id = {gate: index for index, gate in enumerate(GATES)}
assert len(GATES) == 52

@dataclass
class Circuit:
    start: int
    gates: list[str]
    states: list[int]

    @property
    def answer(self):
        return self.states[-1]


def make_circuits(size, seed, depth):
    rng = random.Random(seed)
    return [Circuit(*sample_example(rng, depth)) for _ in range(size)]


train_circuits = make_circuits(TRAIN_SIZE, DATA_SEED, DEPTH)
test_circuits = make_circuits(TEST_SIZE, TEST_SEED, DEPTH)
example = test_circuits[0]
print("Start:", state_text(bits(example.start)))
print("Gates:", example.gates)
print("Gold states:", [state_text(bits(state)) for state in example.states])

Start: 1011
Gates: ['s02', 'c31', 'c31', 't302']
Gold states: ['1011', '1111', '1011', '1001']


## 2. The same prompt, two continuations

Each four-bit state is one semantic token such as `S1000`. These are not separate
bit tokens. Gate strings are also single tokens, giving a vocabulary of 72 tokens.

```text
Prompt for both:   S0000 x0 c01 t012 s03 <SEP>
Outcome target:                             <COLON> S0111 <EOS>
Process target:                             x0 S1000 c01 S1100 t012 S1110 s03 S0111 <COLON> S0111 <EOS>
```

There is no mode token. Separate models learn their own output format. Outcome
supervision includes the delimiter and EOS tokens as well as the final state;
process supervision includes gates, all intermediate states, and the final answer.

In [71]:
class CircuitTokenizer:
    """Map semantic states, gates, and separators to vocabulary indices."""
    def __init__(self, gates, n_states):
        self.n_states = n_states
        self.gates = list(gates)
        self.tokens = [f"S{state:04b}" for state in range(n_states)] + self.gates
        self.tokens += ["<SEP>", "<COLON>", "<EOS>", "<PAD>"]
        self.ids = {token: index for index, token in enumerate(self.tokens)}
        self.sep, self.colon, self.eos, self.pad = [
            self.ids[token] for token in ("<SEP>", "<COLON>", "<EOS>", "<PAD>")]

    def prompt(self, circuit):
        return [circuit.start] + [self.ids[gate] for gate in circuit.gates] + [self.sep]

    def continuation(self, circuit, mode):
        ending = [self.colon, circuit.answer, self.eos]
        if mode == "outcome":
            return ending
        if mode == "process":
            trace = []
            for gate, state in zip(circuit.gates, circuit.states):
                trace.extend([self.ids[gate], state])
            return trace + ending
        raise ValueError(f"Unknown supervision mode: {mode}")

    def decode(self, ids):
        return " ".join(self.tokens[int(index)] for index in ids)


tokenizer = CircuitTokenizer(GATES, N_STATES)
for label, ids in [("Prompt", tokenizer.prompt(example)),
                   ("Outcome", tokenizer.continuation(example, "outcome")),
                   ("Process", tokenizer.continuation(example, "process"))]:
    print(f"{label:8s}: {tokenizer.decode(ids)}")

Prompt  : S1011 s02 c31 c31 t302 <SEP>
Outcome : <COLON> S1001 <EOS>
Process : s02 S1011 c31 S1111 c31 S1011 t302 S1001 <COLON> S1001 <EOS>


## 3. A trainable causal Transformer

```text
Token IDs + positions
          ↓
     embeddings h₀
          ↓
  causal multi-head attention ──→ add h₀
          ↓
       ReLU MLP ────────────────→ add residual
          ↓
   logits over all 72 tokens
          ↓
  choose next token → append → run again
```

At each position the model can attend to that position and earlier positions.
For a next-state prediction after a gate token, a head can retrieve a preceding
state and the MLP can combine it with the gate. Nothing explicitly tells the
learned attention where to look. It must learn from next-token losses.

| Tensor | Shape |
|---|---|
| Token IDs | `[batch, length]` |
| Hidden activations | `[batch, length, d_model]` |
| Attention per head | `[batch, heads, query_position, key_position]` |
| Vocabulary logits | `[batch, length, vocabulary]` |

No transition-matrix parameters, Python gate executor, or oracle correction are
used inside this learned model.

In [72]:
class LearnedOneLayerTransformer(nn.Module):
    """One causal attention layer and one hidden ReLU MLP, with residuals."""
    def __init__(self, vocab_size, max_length, d_model, n_heads, d_ff):
        super().__init__()
        if d_model % n_heads:
            raise ValueError("d_model must be divisible by n_heads")
        self.max_length = max_length
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.position_embedding = nn.Embedding(max_length, d_model)
        self.attention = nn.MultiheadAttention(d_model, n_heads, dropout=0.0, batch_first=True)
        self.mlp = nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
        self.readout = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, ids, return_attention=False):
        length = ids.shape[1]
        if length > self.max_length:
            raise ValueError(f"Input length {length} exceeds {self.max_length}")
        positions = torch.arange(length, device=ids.device)
        hidden = self.token_embedding(ids) + self.position_embedding(positions)[None]
        # True entries are forbidden: no token can see a future token.
        causal_mask = torch.ones(length, length, device=ids.device, dtype=torch.bool).triu(1)
        attended, weights = self.attention(
            hidden, hidden, hidden, attn_mask=causal_mask,
            need_weights=return_attention, average_attn_weights=False,
        )
        hidden = hidden + attended
        hidden = hidden + self.mlp(hidden)
        logits = self.readout(hidden)
        return (logits, weights) if return_attention else logits


torch.manual_seed(MODEL_SEED)
base = LearnedOneLayerTransformer(
    len(tokenizer.tokens), max_length=3 * DEPTH + 5,
    d_model=D_MODEL, n_heads=N_HEADS, d_ff=D_FF,
).to(DEVICE)
print(f"Trainable parameters per learned model: {sum(p.numel() for p in base.parameters()):,}")

Trainable parameters per learned model: 89,856


## 4. Construct a fixed Transformer reference

This reference uses **fixed semantic embeddings**, four dot-product causal
attention heads (three active), a ReLU MLP, and a fixed vocabulary readout.
It generates through exactly the same generic autoregressive loop as the learned
models. The Boolean rule is encoded in weights during initialization, not called
by the forward pass.

- The state head retrieves the previous state at each generated gate position.
- The gate head copies the next gate from the prompt.
- The final head copies the last computed state after `<COLON>`.
- One hidden ReLU unit per `(state, gate)` pair computes
  $\operatorname{ReLU}(\mathbf1_s+\mathbf1_g-1.5)$ and writes the target state.

Routing relies on the fixed-depth output format and absolute positions. Large,
finite attention scores approximate one-hot routing; **greedy token correctness**
is tested below, not exact one-hot probabilities or arbitrary-depth generalization.
The reference has 832 MLP units versus 192 in the default learned model.

In [73]:
class FixedAttentionHead(nn.Module):
    """Ordinary causal dot-product attention with fixed projection buffers."""
    def __init__(self, width, n_positions):
        super().__init__()
        self.register_buffer("query", torch.zeros(n_positions, width))
        self.register_buffer("key", torch.zeros(n_positions, width))
        self.register_buffer("value", torch.zeros(width, width))

    def forward(self, hidden):
        query = hidden @ self.query.T
        key = hidden @ self.key.T
        value = hidden @ self.value.T
        scores = query @ key.transpose(-2, -1) / math.sqrt(self.query.shape[0])
        length = hidden.shape[1]
        mask = torch.ones(length, length, dtype=torch.bool, device=hidden.device).triu(1)
        return scores.masked_fill(mask, -torch.inf).softmax(dim=-1) @ value


class HandcodedProcessTransformer(nn.Module):
    """Constructive fixed-depth attention/MLP solution, encoded entirely in weights."""
    def __init__(self, tokenizer, depth):
        super().__init__()
        states, gates = tokenizer.n_states, len(tokenizer.gates)
        positions = 3 * depth + 4  # Longest prefix before the EOS prediction.
        # Disjoint feature blocks keep the construction readable.
        state = slice(0, states)
        gate = slice(state.stop, state.stop + gates)
        position = slice(gate.stop, gate.stop + positions)
        retrieved_state = slice(position.stop, position.stop + states)
        output_gate = slice(retrieved_state.stop, retrieved_state.stop + gates)
        output_state = slice(output_gate.stop, output_gate.stop + states)
        width = output_state.stop
        self.max_length = positions
        self.heads = nn.ModuleList([FixedAttentionHead(width, positions) for _ in range(4)])
        self.register_buffer("token_features", torch.zeros(len(tokenizer.tokens), width))
        self.register_buffer("position_features", torch.zeros(positions, width))
        self.register_buffer("mlp_in", torch.zeros(states * gates, width))
        self.register_buffer("mlp_bias", torch.full((states * gates,), -1.5))
        self.register_buffer("mlp_out", torch.zeros(width, states * gates))
        self.register_buffer("readout", torch.zeros(len(tokenizer.tokens), width))

        self.token_features[:states, state] = torch.eye(states)
        self.token_features[states:states + gates, gate] = torch.eye(gates)
        self.position_features[:, position] = torch.eye(positions)
        for head in self.heads:
            head.key[:, position] = torch.eye(positions)

        state_head, gate_head, final_head, _ = self.heads
        separator_position = depth + 1
        # Unused routes point to SEP, whose state and gate features are zero.
        state_routes, gate_routes = {}, {}
        for step in range(depth):
            gate_position = depth + 2 + 2 * step
            state_routes[gate_position] = 0 if step == 0 else gate_position - 1
            gate_routes[gate_position - 1] = step + 1
        final_routes = {3 * depth + 2: 3 * depth + 1}
        for head, routes in zip(self.heads[:3], [state_routes, gate_routes, final_routes]):
            for destination in range(separator_position, positions):
                origin = routes.get(destination, separator_position)
                head.query[origin, position.start + destination] = 80.0
        state_head.value[retrieved_state, state] = torch.eye(states)
        gate_head.value[output_gate, gate] = torch.eye(gates)
        final_head.value[output_state, state] = torch.eye(states)

        # Each hidden unit recognizes one pair. This is initialization only.
        for state_id in range(states):
            for gate_id, gate_name in enumerate(tokenizer.gates):
                unit = state_id * gates + gate_id
                self.mlp_in[unit, retrieved_state.start + state_id] = 1
                self.mlp_in[unit, gate.start + gate_id] = 1
                target = phi(state_id, gate_name)
                self.mlp_out[output_state.start + target, unit] = 2
        self.readout[:states, output_state] = 20 * torch.eye(states)
        self.readout[states:states + gates, output_gate] = 20 * torch.eye(gates)
        self.readout[tokenizer.colon, position.start + 3 * depth + 1] = 20
        self.readout[tokenizer.eos, position.start + 3 * depth + 3] = 20

    def forward(self, ids):
        length = ids.shape[1]
        if length > self.max_length:
            raise ValueError("Prefix exceeds the hand-coded routing layout")
        features = self.token_features[ids] + self.position_features[:length][None]
        hidden = features + sum(head(features) for head in self.heads)
        hidden = hidden + F.relu(hidden @ self.mlp_in.T + self.mlp_bias) @ self.mlp_out.T
        return hidden @ self.readout.T


handcoded_model = HandcodedProcessTransformer(tokenizer, DEPTH).to(DEVICE)

### Hand-coded outcome-only Transformer

This reference emits `<COLON> final_state <EOS>`. It computes the intermediate
states internally, using **one causal attention + ReLU MLP block per gate**:

```text
Prompt: S0 g1 g2 g3 g4 <SEP>
Generated tokens:             <COLON> S4 <EOS>
                                 |
                       answer-position hidden state
                       block 1: S0 + g1 → S1
                       block 2: S1 + g2 → S2
                       block 3: S2 + g3 → S3
                       block 4: S3 + g4 → S4 → readout
```

Each block attends to its gate in the prompt. The first also retrieves the initial
state; subsequent blocks read the previous block's computed state. Disjoint feature
slots let residual connections retain intermediate values without overwriting them.
A ReLU unit recognizes `(previous state, gate, answer position)` with threshold 2.5.
The position condition prevents state logits from competing with COLON and EOS.

`phi` is used only to construct fixed MLP weights. Inference performs tensor
operations and standard greedy generation; no hidden call generates a process trace.
Absolute routing assumes the configured depth and valid prompt format. Finite
softmax routing approximates exact selection; we verify generated tokens below.

**Architecture distinction:** the default outcome reference has four blocks, while
the learned models and process reference have one. This supplies a constructive
outcome-only solution, not a capacity-matched control or an impossibility result
for a one-layer outcome model. Fixed weights are buffers, so `.parameters()` counts
zero trainable parameters even though the model stores substantial weights.

In [96]:
class FixedOutcomeBlock(nn.Module):
    """One causal attention/MLP block applies one gate in hidden activations."""
    def __init__(self, tokenizer, width, positions, position_features, token_gate,
                 previous_state, next_state, retrieved_gate, gate_position,
                 answer_position, initial_state=None):
        super().__init__()
        states, gates = tokenizer.n_states, len(tokenizer.gates)
        self.gate_head = FixedAttentionHead(width, positions)
        self.state_head = FixedAttentionHead(width, positions)
        for head in (self.gate_head, self.state_head):
            head.key[:, position_features] = torch.eye(positions)

        # The answer query reads gate t from its fixed position in the prompt.
        self.gate_head.query[gate_position, position_features.start + answer_position] = 80
        self.gate_head.value[retrieved_gate, token_gate] = torch.eye(gates)
        if initial_state is not None:
            # Only block 1 retrieves S0. Later blocks read the previous block's state.
            self.state_head.query[0, position_features.start + answer_position] = 80
            self.state_head.value[previous_state, initial_state] = torch.eye(states)

        self.register_buffer("mlp_in", torch.zeros(states * gates, width))
        self.register_buffer("mlp_bias", torch.full((states * gates,), -2.5))
        self.register_buffer("mlp_out", torch.zeros(width, states * gates))
        for state_id in range(states):
            for gate_id, gate_name in enumerate(tokenizer.gates):
                unit = state_id * gates + gate_id
                self.mlp_in[unit, previous_state.start + state_id] = 1
                self.mlp_in[unit, retrieved_gate.start + gate_id] = 1
                self.mlp_in[unit, position_features.start + answer_position] = 1
                # Encode the Boolean rule once, when constructing weights.
                target = phi(state_id, gate_name)
                self.mlp_out[next_state.start + target, unit] = 2

    def forward(self, hidden):
        hidden = hidden + self.gate_head(hidden) + self.state_head(hidden)
        activation = F.relu(hidden @ self.mlp_in.T + self.mlp_bias)
        return hidden + activation @ self.mlp_out.T


class HandcodedOutcomeTransformer(nn.Module):
    """Fixed-depth causal Transformer that emits only COLON, answer, and EOS.

    Uses one attention/MLP block per gate. Intermediate states live in disjoint
    residual-stream feature blocks, not generated tokens. All weights are fixed
    buffers; there is no Python Boolean execution in forward().
    """
    def __init__(self, tokenizer, depth):
        super().__init__()
        if depth < 1:
            raise ValueError("depth must be positive")
        states, gates = tokenizer.n_states, len(tokenizer.gates)
        positions = depth + 4  # Prefix through the answer, before EOS.
        answer_position = depth + 2  # COLON position predicts the answer.
        self.max_length = positions

        # Semantic input features are separate from computed state features.
        token_state = slice(0, states)
        token_gate = slice(token_state.stop, token_state.stop + gates)
        position = slice(token_gate.stop, token_gate.stop + positions)
        state_slots = [slice(position.stop + step * states,
                             position.stop + (step + 1) * states)
                       for step in range(depth + 1)]
        gate_slots = [slice(state_slots[-1].stop + step * gates,
                            state_slots[-1].stop + (step + 1) * gates)
                      for step in range(depth)]
        width = gate_slots[-1].stop
        self.register_buffer("token_features", torch.zeros(len(tokenizer.tokens), width))
        self.register_buffer("position_features", torch.zeros(positions, width))
        self.register_buffer("readout", torch.zeros(len(tokenizer.tokens), width))
        self.token_features[:states, token_state] = torch.eye(states)
        self.token_features[states:states + gates, token_gate] = torch.eye(gates)
        self.position_features[:, position] = torch.eye(positions)

        self.blocks = nn.ModuleList([
            FixedOutcomeBlock(
                tokenizer, width, positions, position, token_gate,
                state_slots[step], state_slots[step + 1], gate_slots[step],
                gate_position=step + 1, answer_position=answer_position,
                initial_state=token_state if step == 0 else None,
            ) for step in range(depth)
        ])
        self.readout[:states, state_slots[-1]] = 20 * torch.eye(states)
        self.readout[tokenizer.colon, position.start + depth + 1] = 20
        self.readout[tokenizer.eos, position.start + depth + 3] = 20

    def forward(self, ids):
        if ids.shape[1] > self.max_length:
            raise ValueError("Prefix exceeds the fixed outcome routing layout")
        hidden = self.token_features[ids] + self.position_features[:ids.shape[1]][None]
        for block in self.blocks:
            hidden = block(hidden)
        return hidden @ self.readout.T


handcoded_outcome_model = HandcodedOutcomeTransformer(tokenizer, DEPTH).to(DEVICE)

## 5. Generation feeds back predictions—not labels

The same loop serves all models. It chooses the largest **full-vocabulary**
logit; there is no forced gate copying, state-only restriction, or oracle lookup.
EOS stops each sequence. A maximum token budget bounds malformed continuations.

In [74]:
@torch.no_grad()
def generate(model, prompts, max_new_tokens, eos_id):
    """Greedy autoregression; only previously generated tokens are fed back."""
    was_training = model.training
    model.eval()
    ids = prompts.clone()
    finished = torch.zeros(len(ids), dtype=torch.bool, device=ids.device)
    try:
        for _ in range(max_new_tokens):
            next_ids = model(ids)[:, -1].argmax(dim=-1)
            next_ids = torch.where(finished, eos_id, next_ids)
            ids = torch.cat([ids, next_ids[:, None]], dim=1)
            finished |= next_ids == eos_id
            if finished.all():
                break
        return ids
    finally:
        model.train(was_training)


def strip_after_eos(ids, eos_id):
    result = list(map(int, ids))
    return result[:result.index(eos_id) + 1] if eos_id in result else result


example_prompt = torch.tensor([tokenizer.prompt(example)], device=DEVICE)
reference_output = generate(handcoded_model, example_prompt, 2 * DEPTH + 3, tokenizer.eos)
reference_target = tokenizer.prompt(example) + tokenizer.continuation(example, "process")
assert reference_output[0].tolist() == reference_target
print("Fixed Transformer:", tokenizer.decode(reference_output[0]))

Fixed Transformer: S1011 s02 c31 c31 t302 <SEP> s02 S1011 c31 S1111 c31 S1011 t302 S1001 <COLON> S1001 <EOS>


In [ ]:
# Generate only the short answer continuation from the prompt.
outcome_reference_output = generate(handcoded_outcome_model, example_prompt, 3, tokenizer.eos)
expected_outcome = tokenizer.prompt(example) + tokenizer.continuation(example, "outcome")
assert outcome_reference_output[0].tolist() == expected_outcome
print("Hand-coded outcome:", tokenizer.decode(outcome_reference_output[0]))

# Check complete autoregressive continuations on every test circuit in small batches.
for start in range(0, len(test_circuits), 32):
    circuits = test_circuits[start:start + 32]
    prompts = torch.tensor([tokenizer.prompt(c) for c in circuits], device=DEVICE)
    expected = torch.tensor([tokenizer.prompt(c) + tokenizer.continuation(c, "outcome")
                             for c in circuits], device=DEVICE)
    actual = generate(handcoded_outcome_model, prompts, 3, tokenizer.eos)
    assert torch.equal(actual, expected)
print(f"Hand-coded outcome: {len(test_circuits)}/{len(test_circuits)} exact test continuations")

Hand-coded outcome: S1011 s02 c31 c31 t302 <SEP> <COLON> S1001 <EOS>
Hand-coded outcome: 300/300 exact test continuations


## 6. Shifted language-model targets and prompt masking

For a sequence `prompt + continuation`, the input is `full[:-1]` and the target
is `full[1:]`. Target positions predicting prompt tokens are marked `-100` and
ignored by cross-entropy. The position containing `<SEP>` predicts the first
continuation token and **must remain supervised**.

Teacher forcing supplies earlier gold continuation tokens during training.
Free generation in section 5 supplies the model's own tokens instead. Loss is
averaged over supervised continuation tokens, not prompt tokens.

In [75]:
@dataclass
class LanguageBatch:
    inputs: torch.Tensor
    targets: torch.Tensor

    def select(self, indices):
        return LanguageBatch(self.inputs[indices], self.targets[indices])

    def to(self, device):
        return LanguageBatch(self.inputs.to(device), self.targets.to(device))


def encode_dataset(circuits, tokenizer, mode):
    inputs, targets = [], []
    for circuit in circuits:
        prompt = tokenizer.prompt(circuit)
        full = prompt + tokenizer.continuation(circuit, mode)
        inputs.append(full[:-1])
        target = full[1:]
        target[:len(prompt) - 1] = [-100] * (len(prompt) - 1)
        targets.append(target)
    return LanguageBatch(torch.tensor(inputs), torch.tensor(targets))


def language_model_loss(model, batch):
    logits = model(batch.inputs)
    return F.cross_entropy(logits.flatten(0, 1), batch.targets.flatten(), ignore_index=-100)


def make_batch_schedule(size, steps, batch_size, seed):
    rng = random.Random(seed)
    return [[rng.randrange(size) for _ in range(batch_size)] for _ in range(steps)]


training_data = {mode: encode_dataset(train_circuits, tokenizer, mode).to(DEVICE)
                 for mode in ("outcome", "process")}
batch_schedule = make_batch_schedule(TRAIN_SIZE, STEPS, BATCH_SIZE, BATCH_SEED)
for mode, batch in training_data.items():
    print(mode, tuple(batch.inputs.shape), "supervised tokens per circuit:",
          int((batch.targets[0] != -100).sum()))

outcome (8000, 8) supervised tokens per circuit: 3
process (8000, 16) supervised tokens per circuit: 11


## 7. Evaluate complete free-running outputs

**Exact continuation** checks every generated token, including gates, states,
colon, and EOS. **Final answer** parses the state after colon from the generated
text. A correct final answer alone does not imply a correct trace.

The outcome model is scored on its short format, the process model on its longer
format. Their exact-continuation scores have different difficulty. Every model
receives the same test prompts.

In [76]:
@dataclass
class GenerationEvaluation:
    prompts: torch.Tensor
    targets: dict
    answers: list[int]
    depth: int


def make_generation_evaluation(circuits, tokenizer, device):
    return GenerationEvaluation(
        prompts=torch.tensor([tokenizer.prompt(c) for c in circuits], device=device),
        targets={mode: [tokenizer.continuation(c, mode) for c in circuits]
                 for mode in ("outcome", "process")},
        answers=[c.answer for c in circuits], depth=len(circuits[0].gates),
    )


def generated_answer(tokens, tokenizer):
    if tokenizer.colon not in tokens:
        return None
    index = tokens.index(tokenizer.colon) + 1
    if index < len(tokens) and tokens[index] < tokenizer.n_states:
        return tokens[index]
    return None


@torch.no_grad()
def free_run_metrics(model, evaluation, tokenizer, mode):
    budget = 3 if mode == "outcome" else 2 * evaluation.depth + 3
    output = generate(model, evaluation.prompts, budget, tokenizer.eos)
    continuations = output[:, evaluation.prompts.shape[1]:].cpu().tolist()
    predictions = [strip_after_eos(row, tokenizer.eos) for row in continuations]
    targets = evaluation.targets[mode]
    count = len(targets)
    metrics = {
        "exact_continuation": sum(p == t for p, t in zip(predictions, targets)) / count,
        "final_answer": sum(generated_answer(p, tokenizer) == a
                            for p, a in zip(predictions, evaluation.answers)) / count,
    }
    return metrics



# Training accuracy uses a fixed sample, explicitly distinct from the full test set.
TRAIN_EVAL_SIZE = min(300, len(train_circuits))
train_generation_eval = make_generation_evaluation(train_circuits[:TRAIN_EVAL_SIZE], tokenizer, DEVICE)
test_generation_eval = make_generation_evaluation(test_circuits, tokenizer, DEVICE)
fixed_evaluation = free_run_metrics(handcoded_model, test_generation_eval, tokenizer, "process")
assert fixed_evaluation["exact_continuation"] == 1.0
print(f"Fixed Transformer on {len(test_circuits)} test circuits:", fixed_evaluation)

Fixed Transformer on 300 test circuits: {'exact_continuation': 1.0, 'final_answer': 1.0}


## 8. Train and evaluate generated answers

Keep three quantities separate:

- `train_loss`: next-token loss on a fixed subset of up to 256 training circuits.
- `train_answer_accuracy_sample`: autoregressive answer accuracy on the first
  `TRAIN_EVAL_SIZE` training circuits (up to 300), not the entire training set.
- `test_answer_accuracy`: autoregressive answer accuracy on every test circuit.

`test_exact_continuation` additionally checks every output token, including the
whole process trace. Gold tokens are supplied for the loss calculation only;
all accuracy measurements use generated continuations from prompts alone.

Each checkpoint measures the models after that update. Both models start from
independent copies of `base` and use identical sampled circuit indices.

In [77]:
def train_step(model, batch, optimizer):
    """One update using teacher-forced next-token loss."""
    model.train()
    optimizer.zero_grad(set_to_none=True)
    loss = language_model_loss(model, batch)
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate_checkpoint(model, mode, step, loss_sample, train_eval, test_eval, tokenizer):
    """Report loss and actual generated-answer accuracy with explicit split names."""
    train_metrics = free_run_metrics(model, train_eval, tokenizer, mode)
    test_metrics = free_run_metrics(model, test_eval, tokenizer, mode)
    return {
        "step": step,
        "mode": mode,
        "train_loss": language_model_loss(model, loss_sample).item(),
        "train_answer_accuracy_sample": train_metrics["final_answer"],
        "test_answer_accuracy": test_metrics["final_answer"],
        "test_exact_continuation": test_metrics["exact_continuation"],
    }


def train_one_model(base, mode, data, batch_schedule, learning_rate, checkpoints,
                    train_eval, test_eval, tokenizer):
    """Train a fresh copy and return its model and checkpoint history."""
    model = copy.deepcopy(base)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0)
    loss_sample = data.select(slice(0, 256))
    history = [evaluate_checkpoint(model, mode, 0, loss_sample, train_eval, test_eval, tokenizer)]
    for step, indices in enumerate(batch_schedule, 1):
        train_step(model, data.select(indices), optimizer)
        if step in checkpoints:
            row = evaluate_checkpoint(model, mode, step, loss_sample, train_eval, test_eval, tokenizer)
            history.append(row)
            print(f"{mode:7s} step {step:5d}: loss={row['train_loss']:.3f}, "
                  f"train sample answer={row['train_answer_accuracy_sample']:.1%}, "
                  f"test answer={row['test_answer_accuracy']:.1%}", flush=True)
    return model, pd.DataFrame(history)


def run_experiment(base, training_data, batch_schedule, learning_rate, checkpoints,
                   train_eval, test_eval, tokenizer):
    """Compare both supervision methods on identical circuits and test prompts."""
    models, histories = {}, []
    for mode in ("outcome", "process"):
        model, history = train_one_model(
            base, mode, training_data[mode], batch_schedule, learning_rate,
            checkpoints, train_eval, test_eval, tokenizer)
        models[mode] = model
        histories.append(history)
    return models, pd.concat(histories, ignore_index=True)

In [78]:
models, history = run_experiment(
    base, training_data, batch_schedule, LR, CHECKPOINTS,
    train_generation_eval, test_generation_eval, tokenizer,
)
print(f"Training accuracy: fixed sample of {TRAIN_EVAL_SIZE} circuits. "
      f"Test accuracy: all {len(test_circuits)} test circuits.")
display(history)

outcome step   100: loss=0.924, train sample answer=9.3%, test answer=8.3%
outcome step   250: loss=0.883, train sample answer=12.0%, test answer=9.3%
outcome step   500: loss=0.887, train sample answer=10.7%, test answer=8.0%
outcome step 10000: loss=0.217, train sample answer=78.3%, test answer=20.3%
process step   100: loss=0.465, train sample answer=16.0%, test answer=16.7%
process step   250: loss=0.009, train sample answer=98.3%, test answer=99.3%
process step   500: loss=0.001, train sample answer=100.0%, test answer=100.0%
process step 10000: loss=0.000, train sample answer=100.0%, test answer=100.0%
Training accuracy: fixed sample of 300 circuits. Test accuracy: all 300 test circuits.


,step,mode,train_loss,train_answer_accuracy_sample,test_answer_accuracy,test_exact_continuation
0,0,outcome,5.217943e+00,0.000000,0.000000,0.000000
1,100,outcome,9.244630e-01,0.093333,0.083333,0.083333
2,250,outcome,8.825580e-01,0.120000,0.093333,0.093333
3,500,outcome,8.869745e-01,0.106667,0.080000,0.080000
4,10000,outcome,2.174593e-01,0.783333,0.203333,0.203333
5,0,process,4.658970e+00,0.000000,0.000000,0.000000
6,100,process,4.645573e-01,0.160000,0.166667,0.116667
7,250,process,9.181779e-03,0.983333,0.993333,0.993333
8,500,process,5.429616e-04,1.000000,1.000000,1.000000
9,10000,process,1.110813e-07,1.000000,1.000000,1.000000


In [91]:
for name, model in models.items():
    count = sum(p.numel() for p in model.parameters())
    print(f"{name}: {count:,} parameters")

outcome: 89,856 parameters
process: 89,856 parameters


## 9. Animate training loss and generated-answer accuracy

The `FuncAnimation` below uses the checkpoint table directly. It shows training
loss, training-sample versus test answer accuracy, and exact test continuation
accuracy. There are no local gate probes or matrix diagnostics.

The slider advances through saved **training checkpoints**, not generated tokens.
No results are interpolated between checkpoints. Use `CHECKPOINTS` to choose
more frequent measurements; each checkpoint runs autoregressive evaluation.

In [79]:
from matplotlib.animation import FuncAnimation


def animate_training_dynamics(history, interval_ms=600):
    """Return a FuncAnimation of loss and autoregressive train/test accuracy."""
    if interval_ms <= 0:
        raise ValueError("interval_ms must be positive")
    frames = {mode: history[history["mode"] == mode].sort_values("step")
              for mode in ("outcome", "process")}
    steps = frames["outcome"]["step"].tolist()
    if not steps or steps != frames["process"]["step"].tolist():
        raise ValueError("Both models need matching, nonempty checkpoint steps")
    colors = {"outcome": "#d97706", "process": "#0284c7"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), constrained_layout=True)
    heading = fig.suptitle("")
    specifications = [
        ("train_loss", 0, "-", "Training loss (fixed sample)"),
        ("train_answer_accuracy_sample", 1, "--", "Generated answer accuracy"),
        ("test_answer_accuracy", 1, "-", "Generated answer accuracy"),
        ("test_exact_continuation", 2, "-", "Exact test continuation"),
    ]
    lines = {}
    for metric, panel, style, title in specifications:
        ax = axes[panel]
        for mode, color in colors.items():
            split = "train sample" if metric == "train_answer_accuracy_sample" else "test"
            label = f"{mode}: {split}" if panel == 1 else mode
            lines[metric, mode], = ax.plot([], [], style, color=color, marker="o", label=label)
        ax.set(title=title, xlabel="Optimization step", xlim=(0, max(1, steps[-1])))
    axes[0].set(ylabel="Next-token cross-entropy", ylim=(0, max(0.1, history.train_loss.max() * 1.1)))
    for ax in axes[1:]:
        ax.set(ylabel="Accuracy", ylim=(-0.03, 1.03))
    for ax in axes:
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)

    def update_frame(index):
        heading.set_text(f"Transformer training — step {steps[index]}")
        for (metric, mode), line in lines.items():
            data = frames[mode].iloc[:index + 1]
            line.set_data(data["step"], data[metric])
        return [heading, *lines.values()]

    animation = FuncAnimation(fig, update_frame, frames=len(steps),
                              init_func=lambda: update_frame(0), interval=interval_ms,
                              repeat=False, blit=False)
    plt.close(fig)
    return animation

In [80]:
training_animation = animate_training_dynamics(history)
animation_html = training_animation.to_jshtml(default_mode="once")
display(HTML(animation_html))
Path("training_dynamics.html").write_text(
    "<!doctype html><html><head><meta charset='utf-8'><title>Training and test accuracy</title>"
    "</head><body>" + animation_html + "</body></html>", encoding="utf-8")

480392

## 10. Read actual generated test traces

The table displays test examples in their original order, not selected successes.
For each model, the continuation is generated from the prompt alone. Compare it
with the gold trace to see gate-copy errors, wrong intermediate states, or early EOS.
A trained model's trace can be wrong; the table does not replace predictions with labels.

In [ ]:
def show_generated_traces(models, circuits, tokenizer, device, count=3):
    rows = []
    for index, circuit in enumerate(circuits[:count]):
        prompt = torch.tensor([tokenizer.prompt(circuit)], device=device)
        rows.append({"example": index, "model": "Prompt", "text": tokenizer.decode(prompt[0])})
        rows.append({"example": index, "model": "Gold process target",
                     "text": tokenizer.decode(tokenizer.continuation(circuit, "process"))})
        for name, (model, mode) in models.items():
            budget = len(tokenizer.continuation(circuit, mode))
            generated = generate(model, prompt, budget, tokenizer.eos)[0, prompt.shape[1]:]
            rows.append({"example": index, "model": name, "text": tokenizer.decode(generated)})
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(rows))


show_generated_traces({
    "Hand-coded outcome Transformer": (handcoded_outcome_model, "outcome"),
    "Hand-coded Transformer": (handcoded_model, "process"),
    "Outcome-trained": (models["outcome"], "outcome"),
    "Process-trained": (models["process"], "process"),
}, test_circuits, tokenizer, DEVICE)

## Physical Testing

In [97]:
prompt_text = "S1000 x0 c01 t012 s03 <SEP>"

prompt_ids = torch.tensor(
    [[tokenizer.ids[token] for token in prompt_text.split()]],
    dtype=torch.long,
    device=DEVICE,
)

depth = len(prompt_text.split()) - 2  # Exclude initial state and <SEP>.
print("Prompt:", prompt_text)

for mode, model in models.items():
    max_new_tokens = 3 if mode == "outcome" else 2 * depth + 3

    output = generate(model,prompt_ids,max_new_tokens=max_new_tokens,eos_id=tokenizer.eos,)

    continuation = output[0, prompt_ids.shape[1]:]
    print(f"\n{mode.capitalize()} model:")
    print(tokenizer.decode(continuation))

#generate from handcoded_model too 
output = generate(handcoded_model, prompt_ids, max_new_tokens=max_new_tokens, eos_id=tokenizer.eos)
continuation = output[0, prompt_ids.shape[1]:]
print(f"\nHandcoded model:")
print(tokenizer.decode(continuation))


output = generate(handcoded_outcome_model, prompt_ids, max_new_tokens=max_new_tokens, eos_id=tokenizer.eos)
continuation = output[0, prompt_ids.shape[1]:]
print(f"\nHandcoded model:")
print(tokenizer.decode(continuation))


Prompt: S1000 x0 c01 t012 s03 <SEP>

Outcome model:
<COLON> S1000 <EOS>

Process model:
x0 S0000 c01 S0000 t012 S0000 s03 S0000 <COLON> S0000 <EOS>

Handcoded model:
x0 S0000 c01 S0000 t012 S0000 s03 S0000 <COLON> S0000 <EOS>

Handcoded model:
<COLON> S0000 <EOS>


## 11. Final test results

This table reports only autoregressive results on the test circuits:
`test_answer_accuracy` checks the final state, while `test_exact_continuation`
checks the entire expected output. For the process model, that includes every
intermediate gate/state token. The outcome model has a shorter output format.

Use the training-sample and test accuracy curves above to assess a possible
generalization gap. Training loss is not an accuracy percentage. One seed and one
circuit depth do not establish performance on other depths or distributions.
The fixed reference uses a larger structured architecture; the two learned models
share the same architecture and initialization but different output lengths.

In [ ]:
rows = []
for name, model, mode in [
    ("Hand-coded outcome Transformer", handcoded_outcome_model, "outcome"),
    ("Hand-coded process Transformer", handcoded_model, "process"),
    ("Outcome-trained Transformer", models["outcome"], "outcome"),
    ("Process-trained Transformer", models["process"], "process"),
]:
    metrics = free_run_metrics(model, test_generation_eval, tokenizer, mode)
    rows.append({"model": name, "test_answer_accuracy": metrics["final_answer"],
                 "test_exact_continuation": metrics["exact_continuation"]})
final = pd.DataFrame(rows)
display(final)